In [14]:
import sys, os
sys.path.insert(0, "/mnt/scratch/baburish/TPN-training/final/TPN_God")
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

from tensorflow_probability.substrates import jax as tfp
import tensorflow as tf

# Import JAX and require double precision.
import jax.numpy as jnp
import jax
jax.config.update("jax_enable_x64", True)
dtype = jnp.float64

# Other tools.
from scipy.interpolate import griddata
from lib.gupta import precompute_fixed_grid
import time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import pandas as pd
import csv
# Import TriplePandel stuff
from lib.simdata_i3 import I3SimHandler
from lib.geo import center_track_pos_and_time_based_on_data
from lib.gupta_network_eqx_4comp import get_network_eval_v_fn, get_network_eval_v_fn_f32
from lib.experimental_methods import get_vertex_seeds
from lib.linefit import linefit_3d_time_np, linefit_3d_time_jnp
from fitting.llh_scanner import get_scanner
from fitting.llh_fitter import get_fitter
from dom_track_eval import get_eval_network_doms_and_track
from lib.likelihood_conv_mpe_logsumexp_gupta import get_neg_c_triple_gamma_llh, get_neg_c_triple_gamma_llh_optimized
from lib.likelihood_conv_mpe_w_noise_logsumexp_gupta import get_neg_c_triple_gamma_llh_SRT_noise
from astropy.coordinates import SkyCoord

from palettable.cubehelix import Cubehelix
cx = Cubehelix.make(start=0.3, rotation=-0.5, n=16, reverse=False, gamma=1.0,
     	max_light=1.0,max_sat=0.5, min_sat=1.4).get_mpl_colormap()
import astropy.units as u
from helpers import *

dzen = 0.05 # rad
dazi = 0.05 # rad
n_eval = 25 # number of grid points per axes



n_hidden = 96
gupta = True
n_comp = 4

ModuleNotFoundError: No module named 'lib.pulse_extraction_from_i3'

In [5]:
network_path = '/mnt/scratch/baburish/TPN-training/gupta_mixture_jax/new_weights/4comp_no_penalties_w4096batch_tree_start_epoch_255.eqx'
eval_network_v = get_network_eval_v_fn_f32(bpath=network_path, dtype=dtype, n_hidden=n_hidden)
eval_network_doms_and_track = get_eval_network_doms_and_track(eval_network_v, dtype=dtype, gupta=gupta, n_comp=n_comp)

In [6]:
import glob
PATH_TO_INPUT = '/mnt/research/IceCube/Gupta-Reco/22644/tfrecords/ftr/'
META_FILE_NAME = 'meta_ds_22644_from_1000_to_2000_10_to_100TeV.ftr'
PULSES_FILE_NAME = 'pulses_ds_22644_from_1000_to_2000_10_to_100TeV.ftr'

events_meta_file = os.path.join(PATH_TO_INPUT, META_FILE_NAME)
events_pulses_file = os.path.join(PATH_TO_INPUT, PULSES_FILE_NAME)
geo_file = '/mnt/scratch/baburish/TPN-training/TriplePandelReco_JAX/data/icecube/detector_geometry.csv'

events_meta = pd.read_feather(events_meta_file)
events_data = pd.read_feather(events_pulses_file)

In [8]:
outdir = "/mnt/research/IceCube/Gupta-Reco/22644/tfrecords/"
dataset_id = 22644
file_index_start = 1000
file_index_end = 2000

In [16]:
from pulse_extraction_from_i3 import get_pulse_info
from tfrecords_utils import serialize_example


compression_type = ''
options = tf.io.TFRecordOptions(compression_type=compression_type)

sim_handler = I3SimHandler(df_meta = events_meta,
                            df_pulses = events_data,
                            geo_file = geo_file)

write_path = os.path.join(outdir, f"data_ds_{dataset_id}_from_{file_index_start}_to_{file_index_end}_1st_pulse.tfrecord")
with tf.io.TFRecordWriter(write_path, options) as writer:

    # Loop over events, and write to tfrecords file.
    for i in range(len(events_meta)):
        meta, pulses = sim_handler.get_event_data(i)

    # Get dom locations, first hit times, and total charges (for each dom).
        event_data = sim_handler.get_per_dom_summary_from_sim_data(meta, pulses)

        x = event_data[['x', 'y','z','time', 'charge']].to_numpy()
        y = meta[['muon_energy_at_detector', 'q_tot', 'muon_zenith', 'muon_azimuth', 'muon_time',
                      'muon_pos_x', 'muon_pos_y', 'muon_pos_z', 'spline_mpe_zenith',
                      'spline_mpe_azimuth', 'spline_mpe_time', 'spline_mpe_pos_x',
                      'spline_mpe_pos_y', 'spline_mpe_pos_z']].to_numpy()

        writer.write(serialize_example(
                                tf.constant(x, dtype=tf.float64),
                                tf.constant(y, dtype=tf.float64),
                            )
                        )

print(f"stored {len(events_meta)} events in outfile {write_path}")
print(f"stored {i} events in outfile {write_path}")

ModuleNotFoundError: No module named 'icecube'